# 👥 HR Analytics — Employee Attrition: EDA & Summary Report

**Dataset:** IBM HR Analytics Employee Attrition & Performance (1,470 employees × 35 columns)
**Input:** `Raw Data/IBM HR Analytics Employee Attrition & Performance.csv`

**Outputs:**
| File | Content |
|------|---------|
| `Summary/employee_clean.csv` | Cleaned employee-level data (one row per employee) |
| `Summary/summary.csv` | Attrition summary by department & job role |
| `Summary/HR_Attrition_Report.xlsx` | Multi-sheet Excel report |

---

### 🗺️ Notebook Flow
```
 PART A — EXPLORE                        PART B — PREPARE                      PART C — SUMMARISE
 ───────────────────                     ──────────────────────                ─────────────────────────
 1. Setup                                6. Standardise column names           9.  Attrition breakdowns
 2. Load raw data                 ──►    7. Feature engineering        ──►     10. Department × role summary
 3. Structure & data types               8. Drop constant columns              11. Export (CSV + Excel)
 4. Statistical summary                     & export clean data                12. Key findings
 5. Data quality checks
```

## 1. Setup — Libraries, Paths & Display Options

| Library | Used for |
|---------|----------|
| `pandas` | Loading, cleaning and summarising tables |
| `pathlib` | File paths that work on any operating system |
| `openpyxl` | Writing the Excel report (used by pandas behind the scenes) |

> If `openpyxl` isn't installed, uncomment the `%pip` line and run it **once**, then restart the kernel.

In [ ]:
# %pip install openpyxl     # ← run once if openpyxl is missing

import pandas as pd
from pathlib import Path

# ---------- Display settings ----------
pd.set_option("display.max_columns", None)              # show every column
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format      # 2 decimals for all floats

# ---------- Paths ----------
RAW_FILE   = Path("Raw Data") / "IBM HR Analytics Employee Attrition & Performance.csv"
OUTPUT_DIR = Path("Summary")
OUTPUT_DIR.mkdir(exist_ok=True)                         # create output folder if needed

CLEAN_FILE   = OUTPUT_DIR / "employee_clean.csv"        # employee-level clean data
SUMMARY_FILE = OUTPUT_DIR / "summary.csv"               # department × role summary
EXCEL_REPORT = OUTPUT_DIR / "HR_Attrition_Report.xlsx"  # full Excel report

---
# 🅰️ PART A — Exploratory Data Analysis

## 2. Load the Raw Data
Each row is **one employee**. The target we care about is **`Attrition`** (*Yes* = the employee left).

In [ ]:
df = pd.read_csv(RAW_FILE)

print(f"Loaded {df.shape[0]:,} employees × {df.shape[1]} columns")
df.head()

## 3. Structure & Data Types
We check the column list and types to separate **numeric** columns (for statistics)
from **categorical** ones (for group-by breakdowns).

In [ ]:
# Split columns by type — useful for the rest of the analysis
numeric_cols     = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()

print(f"Numeric columns     ({len(numeric_cols)}):", numeric_cols)
print(f"\nCategorical columns ({len(categorical_cols)}):", categorical_cols)

In [ ]:
# Distinct values per categorical column — confirms the categories are clean
for col in categorical_cols:
    print(f"{col:<16} → {sorted(df[col].unique())}")

## 4. Statistical Summary
`describe()` gives the range and spread of every numeric column.
Watch for columns where **min = max** — they carry no information.

In [ ]:
df.describe().T

## 5. Data Quality Checks

| Check | Why it matters |
|-------|----------------|
| Missing values | Gaps must be filled or removed before analysis |
| Duplicate rows | Repeated employees would inflate counts |
| Constant columns | Same value for everyone → no analytical value |
| Target balance | How many employees left vs stayed |

In [ ]:
# ---- Missing values & duplicates ----
missing_total = df.isna().sum().sum()
dup_rows      = df.duplicated().sum()
dup_ids       = df["EmployeeNumber"].duplicated().sum()

print(f"Missing values (all columns) : {missing_total}")
print(f"Duplicate rows               : {dup_rows}")
print(f"Duplicate EmployeeNumbers    : {dup_ids}")

In [ ]:
# ---- Constant columns: only one unique value across all employees ----
constant_cols = [c for c in df.columns if df[c].nunique() == 1]

print("Constant columns:")
for c in constant_cols:
    print(f"  • {c:<15} always = {df[c].iloc[0]}")

In [ ]:
# ---- Target variable: how balanced is attrition? ----
attrition_counts = df["Attrition"].value_counts()
attrition_share  = df["Attrition"].value_counts(normalize=True).mul(100).round(2)

pd.DataFrame({"Employees": attrition_counts, "Share (%)": attrition_share})

> **💡 EDA takeaways**
> - The data is **complete**: no missing values and no duplicates.
> - `EmployeeCount`, `Over18` and `StandardHours` are **constant**, so we drop them in Part B.
> - The target is **imbalanced**: 237 of 1,470 employees left (**~16%**).

---
# 🅱️ PART B — Data Preparation

## 6. Standardise Column Names
We convert `CamelCase` to `snake_case` (e.g. `MonthlyIncome` → `monthly_income`) so names are
consistent and easy to type in Python, SQL and Power BI.

In [ ]:
df.columns = (
    df.columns.str.strip()
    .str.replace(r"(?<!^)(?=[A-Z])", "_", regex=True)   # insert _ before each capital letter
    .str.lower()
)

# Constant column names must be converted the same way so the drop step works later
constant_cols = [
    pd.Series([c]).str.replace(r"(?<!^)(?=[A-Z])", "_", regex=True).str.lower()[0]
    for c in constant_cols
]

print("Constant columns (snake_case):", constant_cols)
df.columns.tolist()

## 7. Feature Engineering

| New column | Logic | Purpose |
|-----------|-------|---------|
| `attrition_flag` | 1 if `attrition == "Yes"`, else 0 | Lets us **sum** exits and **average** attrition rate |
| `age_group` | Age bins: 18-25, 26-35, 36-45, 46-55, 56+ | Compare attrition across life stages |
| `tenure_group` | Years at company: 0-1, 2-5, 6-10, 11+ | Spot early-tenure exits |
| `income_band` | Monthly income split into 4 equal-sized groups | Compare pay levels |
| `experience_gap` | Total working years − years at company | Experience gained **before** joining |

In [ ]:
# ---- Target as a number (1 = left, 0 = stayed) ----
df["attrition_flag"] = df["attrition"].eq("Yes").astype("int8")

# ---- Age groups ----
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 55, float("inf")],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"],
)

# ---- Tenure groups (bins are right-inclusive: -1 < x <= 1 → "0-1 years") ----
df["tenure_group"] = pd.cut(
    df["years_at_company"],
    bins=[-1, 1, 5, 10, float("inf")],
    labels=["0-1 years", "2-5 years", "6-10 years", "11+ years"],
)

# ---- Income quartiles (each band holds ~25% of employees) ----
df["income_band"] = pd.qcut(
    df["monthly_income"], q=4, labels=["Low", "Medium", "High", "Very High"]
)

# ---- Prior experience outside the current company ----
df["experience_gap"] = df["total_working_years"] - df["years_at_company"]

df[["age", "age_group", "years_at_company", "tenure_group",
    "monthly_income", "income_band", "experience_gap", "attrition_flag"]].head()

## 8. Drop Constant Columns & Export Clean Data

> **🛠️ Fix from the previous version:** the old code tried to drop `employeecount` and
> `standardhours`, but after renaming the columns are called `employee_count` and
> `standard_hours`. Because of `errors="ignore"`, nothing failed, but **2 of the 3 columns were never dropped**.
> We now use the names found in Step 5.

In [ ]:
cols_before = df.shape[1]
df = df.drop(columns=constant_cols)

print(f"Dropped {cols_before - df.shape[1]} constant columns: {constant_cols}")
print(f"Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Guard: make sure the drop really worked
assert not set(constant_cols) & set(df.columns), "Constant columns still present!"

In [ ]:
# Save the employee-level clean data (separate file, so the summary never overwrites it)
df.to_csv(CLEAN_FILE, index=False)
print("✅ Saved:", CLEAN_FILE.resolve())

---
# 🅲 PART C — Attrition Summary

## 9. Attrition Breakdowns
A helper function computes, for any column: **employees**, **exits** and **attrition rate %**,
sorted from highest to lowest rate. This shows which groups are most at risk.

In [ ]:
OVERALL_RATE = df["attrition_flag"].mean() * 100
print(f"Overall attrition rate: {OVERALL_RATE:.2f}%")

def attrition_by(column):
    """Employees, exits and attrition rate (%) for each value of `column`."""
    out = (
        df.groupby(column, observed=True)
        .agg(employees=("attrition_flag", "size"),
             exits=("attrition_flag", "sum"))
    )
    out["attrition_rate_pct"] = (out["exits"] / out["employees"] * 100).round(2)
    out["vs_overall_pct_pts"] = (out["attrition_rate_pct"] - OVERALL_RATE).round(2)  # + = above average
    return out.sort_values("attrition_rate_pct", ascending=False)

### 9.1 Over-time, travel & marital status

In [ ]:
display(attrition_by("over_time"))
display(attrition_by("business_travel"))
display(attrition_by("marital_status"))

### 9.2 Age, tenure & income

In [ ]:
display(attrition_by("age_group"))
display(attrition_by("tenure_group"))
display(attrition_by("income_band"))

### 9.3 Satisfaction & work-life balance (1 = low, 4 = high)

In [ ]:
display(attrition_by("job_satisfaction"))
display(attrition_by("environment_satisfaction"))
display(attrition_by("work_life_balance"))

### 9.4 Stayers vs Leavers — numeric comparison
Average values for employees who **stayed** vs **left**. Large differences point to drivers worth investigating.

In [ ]:
compare_cols = ["age", "monthly_income", "years_at_company", "total_working_years",
                "distance_from_home", "num_companies_worked", "years_since_last_promotion",
                "job_satisfaction", "environment_satisfaction", "work_life_balance"]

stay_vs_leave = df.groupby("attrition")[compare_cols].mean().T
stay_vs_leave["difference"] = stay_vs_leave["Yes"] - stay_vs_leave["No"]
stay_vs_leave.sort_values("difference")

## 10. Department × Job Role Summary
The main summary table: one row per **department + job role** with headcount, exits,
average income, average tenure and attrition rate.

In [ ]:
summary_df = (
    df.groupby(["department", "job_role"])
    .agg(
        total_employees=("employee_number", "count"),     # headcount
        total_exits=("attrition_flag", "sum"),            # employees who left
        avg_monthly_income=("monthly_income", "mean"),    # average pay
        avg_tenure=("years_at_company", "mean"),          # average years at company
    )
    .reset_index()
)

# Attrition rate (%) per role
summary_df["attrition_rate_pct"] = (
    summary_df["total_exits"] / summary_df["total_employees"] * 100
).round(2)

# Round averages for a cleaner report
summary_df[["avg_monthly_income", "avg_tenure"]] = summary_df[["avg_monthly_income", "avg_tenure"]].round(2)

# Sanity check: headcount and exits must add up to the full dataset
assert summary_df["total_employees"].sum() == len(df)
assert summary_df["total_exits"].sum() == df["attrition_flag"].sum()

summary_df.sort_values("attrition_rate_pct", ascending=False)

In [ ]:
# Department-level roll-up
dept_summary = attrition_by("department")
dept_summary

## 11. Export — CSV & Excel Report

| Sheet | Content |
|-------|---------|
| **Role Summary** | Department × job role table (step 10) |
| **Department Summary** | Department roll-up |
| **Attrition Drivers** | All breakdowns from step 9 stacked in one sheet |
| **Stay vs Leave** | Numeric averages for stayers vs leavers |
| **Clean Data** | Employee-level clean dataset |

In [ ]:
# ---- CSV used by Power BI / downstream analysis ----
summary_df.to_csv(SUMMARY_FILE, index=False)
print("✅ Saved:", SUMMARY_FILE.resolve())

# ---- Stack every breakdown into one long table: [factor, group, employees, exits, rate, vs overall] ----
driver_cols = ["over_time", "business_travel", "marital_status", "age_group", "tenure_group",
               "income_band", "job_satisfaction", "environment_satisfaction", "work_life_balance"]

attrition_drivers = pd.concat(
    [attrition_by(c).reset_index().rename(columns={c: "group"}).assign(factor=c) for c in driver_cols],
    ignore_index=True,
)
attrition_drivers["group"] = attrition_drivers["group"].astype(str)
attrition_drivers = attrition_drivers[["factor", "group", "employees", "exits",
                                       "attrition_rate_pct", "vs_overall_pct_pts"]]

# ---- Write the Excel workbook ----
sheets = {
    "Role Summary":       summary_df,
    "Department Summary": dept_summary.reset_index(),
    "Attrition Drivers":  attrition_drivers,
    "Stay vs Leave":      stay_vs_leave.reset_index(names="metric"),
    "Clean Data":         df,
}
with pd.ExcelWriter(EXCEL_REPORT, engine="openpyxl") as writer:
    for name, table in sheets.items():
        table.to_excel(writer, sheet_name=name, index=False)

print("✅ Saved:", EXCEL_REPORT.resolve())
for name, table in sheets.items():
    print(f"   • {name:<20} {len(table):>5,} rows")

## 12. Key Findings
The cell below prints the main findings straight from the data, so they stay correct if the data changes.

In [ ]:
roles = summary_df.sort_values("attrition_rate_pct", ascending=False)
top, low = roles.iloc[0], roles.iloc[-1]
ot = attrition_by("over_time")["attrition_rate_pct"]
ten = attrition_by("tenure_group")["attrition_rate_pct"]
inc = attrition_by("income_band")["attrition_rate_pct"]

print("📌 KEY FINDINGS")
print(f"• Overall attrition: {df['attrition_flag'].sum()} of {len(df):,} employees ({OVERALL_RATE:.2f}%)")
print(f"• Highest-risk role: {top['job_role']} ({top['department']}) at {top['attrition_rate_pct']:.2f}%")
print(f"• Lowest-risk role : {low['job_role']} ({low['department']}) at {low['attrition_rate_pct']:.2f}%")
print(f"• Over-time: {ot.get('Yes', float('nan')):.2f}% attrition vs {ot.get('No', float('nan')):.2f}% without")
print(f"• Tenure: highest attrition in '{ten.idxmax()}' ({ten.max():.2f}%)")
print(f"• Income: '{inc.idxmax()}' band has the highest attrition ({inc.max():.2f}%)")

### ✅ Summary from the department × role table
- **Sales Representatives** have the highest attrition (**~40%**), followed by **Laboratory Technicians (~24%)** and **HR staff (~23%)**.
- **Managers and Research Directors** rarely leave (**0–5.6%**). These roles have the **highest pay (~16–18K)** and the **longest tenure (11–16 years)**.
- **The roles with the most exits are the lowest-paid roles with the shortest tenure.** Sales Representatives, Laboratory Technicians and Research Scientists each average roughly 2.6–3.2K a month and 3–5 years at the company. Pay and early tenure are the first places to look.

### ➡️ Next steps
- Build a Power BI dashboard from `Summary/employee_clean.csv` and `Summary/summary.csv`
- Test which factors matter most with a simple classification model (e.g. logistic regression)
- Focus retention actions on high-risk groups: Sales Representatives, overtime workers, employees in their first years